In [0]:
%python
API_URL = "https://data-api.binance.vision/api/v3/klines"

SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]

INTERVAL = "1h"

BATCH_SIZE = 1000

TARGET_TABLE = "workspace.crypto_bronze.market_candles_v2"

In [0]:
%python

import requests
import time

def fetch_candles(symbol, batches=2):
    all_candles = []
    end_time = None

    for batch in range(batches):
        params = {
            "symbol": symbol,
            "interval": INTERVAL,
            "limit": BATCH_SIZE
        }

        if end_time is not None:
            params["endTime"] = end_time

        response = requests.get(
            API_URL,
            params=params,
            timeout=30
        )
        response.raise_for_status()

        candles = response.json()

        if not candles:
            break

        all_candles.extend(candles)

        end_time = candles[0][0] - 1

        print(
            f"{symbol} | Batch {batch + 1} | "
            f"Fetched: {len(candles)}"
        )

        if len(candles) < BATCH_SIZE:
            break

        time.sleep(0.2)

    return all_candles


raw_data = {}

for symbol in SYMBOLS:
    raw_data[symbol] = fetch_candles(symbol)

print("\nTotal records:")
for symbol, candles in raw_data.items():
    print(symbol, len(candles))


In [0]:
%python

from pyspark.sql import types as T
from pyspark.sql import functions as F

# Define the structure of each candle
schema = T.StructType([
    T.StructField("symbol", T.StringType(), False),
    T.StructField("interval", T.StringType(), False),
    T.StructField("open_time_ms", T.LongType(), False),
    T.StructField("open", T.StringType(), True),
    T.StructField("high", T.StringType(), True),
    T.StructField("low", T.StringType(), True),
    T.StructField("close", T.StringType(), True),
    T.StructField("volume", T.StringType(), True),
    T.StructField("close_time_ms", T.LongType(), True)
])

# Prepare rows for Spark
rows = []

for symbol, candles in raw_data.items():
    for candle in candles:
        rows.append((
            symbol,
            INTERVAL,
            candle[0],
            candle[1],
            candle[2],
            candle[3],
            candle[4],
            candle[5],
            candle[6]
        ))

# Create Spark DataFrame
spark_df = spark.createDataFrame(rows, schema=schema)

# Add ingestion timestamp
spark_df = spark_df.withColumn(
    "ingested_at",
    F.current_timestamp()
)

print("Total rows:", spark_df.count())
spark_df.printSchema()

display(spark_df.limit(10))


In [0]:
%python

from pyspark.sql import functions as F

# 1. Check NULL values in key columns
null_count = spark_df.filter(
    F.col("symbol").isNull() |
    F.col("interval").isNull() |
    F.col("open_time_ms").isNull()
).count()

# 2. Check duplicate candles
duplicate_count = (
    spark_df
    .groupBy("symbol", "interval", "open_time_ms")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 3. Check invalid prices
invalid_prices = spark_df.filter(
    F.col("close").cast("decimal(20,8)").isNull() |
    (F.col("close").cast("decimal(20,8)") <= 0)
).count()

print("NULL keys:", null_count)
print("Duplicate keys:", duplicate_count)
print("Invalid close prices:", invalid_prices)

assert null_count == 0, "NULL keys detected"
assert duplicate_count == 0, "Duplicate candles detected"
assert invalid_prices == 0, "Invalid close prices detected"

print("Data Quality checks passed!")


In [0]:
%python

TARGET_TABLE = "workspace.crypto_bronze.market_candles_v2"

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.crypto_bronze
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    symbol STRING,
    interval STRING,
    open_time_ms BIGINT,
    open STRING,
    high STRING,
    low STRING,
    close STRING,
    volume STRING,
    close_time_ms BIGINT,
    ingested_at TIMESTAMP
)
USING DELTA
""")

print("Bronze Delta table is ready!")


In [0]:
%python

from delta.tables import DeltaTable

# Get existing Delta table
target = DeltaTable.forName(spark, TARGET_TABLE)

# Merge incoming data into Bronze
(
    target.alias("target")
    .merge(
        spark_df.alias("source"),
        """
        target.symbol = source.symbol
        AND target.interval = source.interval
        AND target.open_time_ms = source.open_time_ms
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE completed!")


In [0]:
%sql
SELECT
    symbol,
    COUNT(*) AS total_candles,
    COUNT(DISTINCT open_time_ms) AS unique_candles,
    MIN(timestamp_millis(open_time_ms)) AS first_candle,
    MAX(timestamp_millis(open_time_ms)) AS last_candle
FROM workspace.crypto_bronze.market_candles_v2
GROUP BY symbol
ORDER BY symbol;

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM workspace.crypto_bronze.market_candles_v2;